# 🔍 Pertemuan 3 — Data Quality & Data Cleaning
### Mata Kuliah: Python Machine Learning

---

## Tujuan Pembelajaran

Setelah pertemuan ini, kamu bisa:
1. Menjelaskan apa itu data kotor dan dampaknya terhadap model ML
2. Mendeteksi missing values, duplikat, dan outlier menggunakan Pandas
3. Menangani setiap jenis masalah data dengan strategi yang tepat
4. Memvalidasi hasil cleaning secara sederhana

---

## Dataset: Titanic 🚢

Kita akan menggunakan dataset Titanic — salah satu dataset paling terkenal di dunia ML.  
Dataset ini berisi data **891 penumpang** kapal Titanic beserta apakah mereka selamat atau tidak.

| Kolom | Keterangan | Tipe |
|-------|-----------|------|
| PassengerId | ID unik penumpang | int |
| Survived | Selamat? (0=Tidak, 1=Ya) | int |
| Pclass | Kelas tiket (1, 2, 3) | int |
| Name | Nama penumpang | str |
| Sex | Jenis kelamin | str |
| Age | Umur | float |
| SibSp | Jumlah saudara/pasangan di kapal | int |
| Parch | Jumlah orang tua/anak di kapal | int |
| Ticket | Nomor tiket | str |
| Fare | Harga tiket | float |
| Cabin | Nomor kabin | str |
| Embarked | Pelabuhan keberangkatan (C, Q, S) | str |

---

In [ ]:
# ============================================================
# SETUP — Jalankan cell ini pertama kali
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi tampilan
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

print("✅ Setup selesai!")

---

## BAGIAN 1: Load & Inspeksi Awal

> **Aturan #1 dalam analisis data:** Kenali datamu sebelum melakukan apapun.

In [ ]:
# Load dataset dari seaborn (sudah built-in, tidak perlu download)
df = sns.load_dataset('titanic')

print(f"✅ Dataset berhasil dimuat!")
print(f"Ukuran data: {df.shape[0]} baris x {df.shape[1]} kolom")

In [ ]:
# Lihat 5 baris pertama
df.head()

In [ ]:
# Lihat 5 baris terakhir
df.tail()

In [ ]:
# Informasi tipe data setiap kolom
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

In [ ]:
# ✏️ DISKUSI: Dari df.info() dan df.describe() di atas,
# apa yang sudah kamu perhatikan?
# Tulis pengamatanmu di sini sebagai komentar:

# 1. ...
# 2. ...
# 3. ...

---

## BAGIAN 2: Deteksi Masalah Data

### 2A. Missing Values (Nilai Hilang)

Missing values terjadi ketika data tidak tersedia atau tidak diisi.  
Di Pandas, nilai kosong direpresentasikan sebagai `NaN` (Not a Number).

In [ ]:
# Cek jumlah missing value per kolom
missing = df.isnull().sum()
missing_persen = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_persen
})

# Hanya tampilkan kolom yang ada missing value
missing_df[missing_df['Jumlah Missing'] > 0]

In [ ]:
# Visualisasi missing values
fig, ax = plt.subplots(figsize=(10, 4))

kolom_missing = missing_df[missing_df['Jumlah Missing'] > 0]
bars = ax.bar(kolom_missing.index, kolom_missing['Persentase (%)'], 
              color=['#e74c3c' if p > 50 else '#f39c12' if p > 10 else '#3498db' 
                     for p in kolom_missing['Persentase (%)']])

ax.set_title('Persentase Missing Values per Kolom', fontsize=14, pad=15)
ax.set_ylabel('Persentase Missing (%)')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='Batas 50%')
ax.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='Batas 10%')

for bar, pct in zip(bars, kolom_missing['Persentase (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{pct}%', ha='center', fontsize=11)

ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Coba install dan gunakan missingno untuk visualisasi yang lebih informatif
try:
    import missingno as msno
    msno.matrix(df, figsize=(12, 5), fontsize=12)
    plt.title('Pola Missing Values (putih = data hilang)', pad=15)
    plt.show()
except ImportError:
    print("Install dengan: pip install missingno")

---

### 2B. Data Duplikat

In [ ]:
# Cek duplikat
jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat: {jumlah_duplikat}")

# Tampilkan baris yang duplikat (jika ada)
if jumlah_duplikat > 0:
    print("\nBaris duplikat:")
    print(df[df.duplicated(keep=False)])
else:
    print("✅ Tidak ada data duplikat!")

### 2C. Outlier

Outlier adalah nilai yang **jauh berbeda** dari nilai lainnya.  
Bisa jadi kesalahan input, atau memang data ekstrem yang valid.

In [ ]:
# Visualisasi outlier dengan boxplot
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

kolom_numerik = ['age', 'fare', 'sibsp']
warna = ['#3498db', '#e74c3c', '#2ecc71']

for ax, kolom, warna_col in zip(axes, kolom_numerik, warna):
    ax.boxplot(df[kolom].dropna(), patch_artist=True,
               boxprops=dict(facecolor=warna_col, alpha=0.6))
    ax.set_title(f'Distribusi: {kolom}', fontsize=12)
    ax.set_ylabel('Nilai')

plt.suptitle('Deteksi Outlier dengan Boxplot', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Deteksi outlier dengan metode IQR (Inter-Quartile Range)
def deteksi_outlier_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR
    jumlah_outlier = ((series < batas_bawah) | (series > batas_atas)).sum()
    return batas_bawah, batas_atas, jumlah_outlier

print(f"{'Kolom':<10} {'Batas Bawah':>15} {'Batas Atas':>12} {'Jml Outlier':>12}")
print("-" * 55)

for kolom in ['age', 'fare']:
    bb, ba, jml = deteksi_outlier_iqr(df[kolom].dropna())
    print(f"{kolom:<10} {bb:>15.2f} {ba:>12.2f} {jml:>12}")

### 2D. Inkonsistensi Data Kategorikal

In [ ]:
# Cek nilai unik pada kolom kategorikal
kolom_kategori = ['sex', 'embarked', 'who', 'class']

for kolom in kolom_kategori:
    print(f"\n📋 Kolom '{kolom}':")
    print(df[kolom].value_counts(dropna=False))

---

## BAGIAN 3: Strategi Penanganan Missing Values

Tidak ada satu solusi untuk semua kasus. Gunakan tabel ini sebagai panduan:

| Kondisi | Strategi | Alasan |
|---------|----------|--------|
| Missing > 50% | Drop kolom | Terlalu banyak data hilang, tidak bisa diandalkan |
| Numerik, distribusi normal | Isi dengan **mean** | Tidak menggeser distribusi |
| Numerik, ada outlier/skewed | Isi dengan **median** | Lebih robust terhadap nilai ekstrem |
| Kategorikal | Isi dengan **modus** (nilai paling sering) | Pertahankan distribusi kategori |
| Data penting, sedikit missing | **Drop baris** | Lebih aman daripada imputasi yang salah |

In [ ]:
# Buat salinan dataframe agar data asli tidak berubah
df_clean = df.copy()

print("Missing SEBELUM cleaning:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

In [ ]:
# Kolom 'deck' memiliki missing > 77% — DROP kolom ini
df_clean = df_clean.drop(columns=['deck'])
print(f"✅ Kolom 'deck' dihapus (77% missing).")

In [ ]:
# Kolom 'age' (numerik, sedikit skewed) — isi dengan MEDIAN
median_age = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(median_age)

print(f"✅ Kolom 'age' diisi dengan median: {median_age:.1f} tahun")

In [ ]:
# Kolom 'embarked' (kategorikal) — isi dengan MODUS
modus_embarked = df_clean['embarked'].mode()[0]
df_clean['embarked'] = df_clean['embarked'].fillna(modus_embarked)

print(f"✅ Kolom 'embarked' diisi dengan modus: '{modus_embarked}'")

In [ ]:
# Kolom 'embark_town' (1 baris missing) — isi dengan modus
modus_town = df_clean['embark_town'].mode()[0]
df_clean['embark_town'] = df_clean['embark_town'].fillna(modus_town)

print(f"✅ Kolom 'embark_town' diisi: '{modus_town}'")

In [ ]:
# Verifikasi: cek missing value setelah cleaning
sisa_missing = df_clean.isnull().sum()
print("\nMissing SETELAH cleaning:")
if sisa_missing.sum() == 0:
    print("✅ Tidak ada missing values tersisa!")
else:
    print(sisa_missing[sisa_missing > 0])

---

## BAGIAN 4: Penanganan Duplikat & Outlier

In [ ]:
# Drop duplikat (jika ada)
baris_sebelum = len(df_clean)
df_clean = df_clean.drop_duplicates()
baris_sesudah = len(df_clean)

print(f"Baris sebelum: {baris_sebelum}")
print(f"Baris sesudah: {baris_sesudah}")
print(f"Duplikat dihapus: {baris_sebelum - baris_sesudah}")

In [ ]:
# Handling outlier pada 'fare' dengan capping (batas atas)
# Strategi: nilai di atas batas IQR diganti dengan nilai batas
Q1 = df_clean['fare'].quantile(0.25)
Q3 = df_clean['fare'].quantile(0.75)
IQR = Q3 - Q1
batas_atas_fare = Q3 + 1.5 * IQR

print(f"Batas atas fare: {batas_atas_fare:.2f}")
print(f"Nilai fare maksimum sebelum: {df_clean['fare'].max():.2f}")

df_clean['fare'] = df_clean['fare'].clip(upper=batas_atas_fare)

print(f"Nilai fare maksimum sesudah: {df_clean['fare'].max():.2f}")
print("✅ Outlier fare telah di-cap.")

---

## BAGIAN 5: Validasi Hasil Cleaning

In [ ]:
# Perbandingan sebelum dan sesudah
print("=" * 50)
print("PERBANDINGAN SEBELUM vs SESUDAH CLEANING")
print("=" * 50)

print(f"{'Aspek':<30} {'Sebelum':>10} {'Sesudah':>10}")
print("-" * 50)
print(f"{'Jumlah Baris':<30} {df.shape[0]:>10} {df_clean.shape[0]:>10}")
print(f"{'Jumlah Kolom':<30} {df.shape[1]:>10} {df_clean.shape[1]:>10}")
print(f"{'Total Missing Values':<30} {df.isnull().sum().sum():>10} {df_clean.isnull().sum().sum():>10}")
print(f"{'Rata-rata Age':<30} {df['age'].mean():>10.2f} {df_clean['age'].mean():>10.2f}")
print(f"{'Fare Maksimum':<30} {df['fare'].max():>10.2f} {df_clean['fare'].max():>10.2f}")

In [ ]:
# Visualisasi distribusi age sebelum vs sesudah
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['age'].dropna(), bins=30, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribusi Age — SEBELUM Cleaning', fontsize=12)
axes[0].set_xlabel('Umur')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(df['age'].median(), color='black', linestyle='--', label=f'Median: {df["age"].median():.1f}')
axes[0].legend()

axes[1].hist(df_clean['age'], bins=30, color='#2ecc71', alpha=0.7, edgecolor='white')
axes[1].set_title('Distribusi Age — SESUDAH Cleaning', fontsize=12)
axes[1].set_xlabel('Umur')
axes[1].set_ylabel('Frekuensi')
axes[1].axvline(df_clean['age'].median(), color='black', linestyle='--', label=f'Median: {df_clean["age"].median():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Simpan hasil cleaning untuk digunakan di Pertemuan 4
df_clean.to_csv('titanic_cleaned.csv', index=False)
print("✅ Data bersih disimpan sebagai 'titanic_cleaned.csv'")
print(f"   Total baris: {len(df_clean)} | Total kolom: {len(df_clean.columns)}")

---

## 🎯 Latihan Mandiri

Kerjakan latihan berikut untuk memperkuat pemahaman:

In [ ]:
# LATIHAN 1:
# Buat dataset 'kotor' buatan sendiri dan bersihkan!

data_kotor = {
    'nama': ['Andi', 'Budi', 'Andi', 'Cici', 'Dedi', None, 'Eka'],
    'umur': [20, 999, 20, 21, 18, 22, 19],       # ada outlier (999)
    'kota': ['Jakarta', 'jakarta', 'Jakarta',      # ada inkonsistensi kapital
             'Surabaya', 'bandung', 'Bandung', 'JAKARTA'],
    'nilai': [85, 70, 85, None, 60, 75, 80]        # ada missing
}

df_kotor = pd.DataFrame(data_kotor)
print("Dataset kotor:")
df_kotor

In [ ]:
# ✏️ Tugasmu:
# 1. Hapus baris duplikat
# 2. Handle outlier pada kolom 'umur' (nilai 999 tidak masuk akal)
# 3. Standarisasi kolom 'kota' (semuanya Title Case)
# 4. Isi missing value pada kolom 'nilai' dan 'nama'
# 5. Tampilkan hasil akhirnya

df_latihan = df_kotor.copy()

# Tulis kode di sini:
# Step 1 - Hapus duplikat

# Step 2 - Handle outlier umur

# Step 3 - Standarisasi kota

# Step 4 - Isi missing values

# Step 5 - Tampilkan hasil


---

## 📝 Ringkasan Pertemuan 3

| Masalah | Cara Deteksi | Cara Tangani |
|---------|-------------|-------------|
| Missing Values | `df.isnull().sum()` | Drop kolom (>50%), fillna median/modus, dropna |
| Duplikat | `df.duplicated().sum()` | `df.drop_duplicates()` |
| Outlier | Boxplot, IQR | Capping, transformasi, drop |
| Inkonsistensi | `df['col'].value_counts()` | `.str.title()`, `.str.strip()`, `.replace()` |

**Selanjutnya → Pertemuan 4: Data Preparation for Machine Learning**